In [ ]:
## perterbation experiments for dissecting hepg2 and k562 regulatory code 

In [1]:
import os, pickle, numpy as np

REPO = '/grid/wsbs/home_norepl/pmantill/Virtual_Experiments/Hippo_axis/Hippo_dependency_mpra'
lib_path = os.path.join(REPO, 'virtual_perterbations', 'libraries', 'hippo_focused_library.pkl')

with open(lib_path, 'rb') as f:
    lib = pickle.load(f)

df = lib['df']
motif_hits = lib['motif_hits']
eigen_results = lib['eigen_results']
focus_tfs = lib['focus_tfs']
hippo_tfs = lib['hippo_tfs']
sig_tfs = lib['sig_tfs']

print(f'Loaded {len(df)} sequences')
print(f'  score > 0 (same-same): {(df.score > 0).sum()}')
print(f'  score < 0 (same-diff): {(df.score < 0).sum()}')
print(f'  Focus TFs: {len(focus_tfs)}')
print(f'  Hippo TFs: {hippo_tfs}')
print(f'  Sig TFs: {sig_tfs}')
print(f'\nColumns: {list(df.columns)}')
print(f'Attribution maps: index into {lib["attr_npz_path"]} with df.idx')

Loaded 29545 sequences
  score > 0 (same-same): 27127
  score < 0 (same-diff): 2418
  Focus TFs: 248
  Hippo TFs: ['BATF::JUN', 'FOS', 'FOS::JUND', 'Fosb', 'JUN', 'JUN::JUNB', 'JUNB', 'JUND', 'Jun', 'RUNX2', 'RUNX3', 'Runx1', 'STAT1', 'STAT1::STAT2', 'STAT3', 'Stat2', 'Stat4', 'Stat5a', 'Stat5a::Stat5b', 'Stat5b', 'Stat6', 'TEAD1', 'TEAD2', 'TEAD3', 'TEAD4']
  Sig TFs: ['ATF3', 'ATF4', 'ATF7', 'Ahr::Arnt', 'BACH1', 'BATF3', 'BCL11A', 'BHLHE22', 'Bach1::Mafk', 'Banp', 'CEBPA', 'CEBPD', 'CEBPG', 'CGGBP1', 'CREB3', 'CREM', 'CTCFL', 'Creb5', 'Dmrt1', 'E2F6', 'EGR2', 'EGR3', 'ELF1', 'ELF2', 'ELF4', 'ELK1', 'ELK1::SREBF2', 'ELK3', 'ELK4', 'ERF::NHLH1', 'ESRRA', 'ESRRB', 'ETS2', 'ETV1', 'ETV2', 'ETV5', 'ETV6', 'EWSR1-FLI1', 'Erg', 'FEV', 'FLI1', 'FOS', 'FOXA1', 'FOXB1', 'FOXD1', 'FOXD2', 'FOXE1', 'FOXG1', 'FOXI1', 'FOXN3', 'FOXO1::FLI1', 'FOXO6', 'Foxj3', 'GABPA', 'GATA1::TAL1', 'GATA2', 'GATA4', 'GATA6', 'GLI3', 'Gata3', 'HNF1A', 'HNF1B', 'HNF4A', 'HNF4G', 'HOXB13', 'Hnf1A', 'IKZF1', 'IKZF2'

In [2]:
## Select same-diff examples from extremes

# Target TF families
target_families = {
    'HNF': ['HNF1A', 'HNF1B', 'HNF4A', 'HNF4G'],
    'STAT': ['STAT1', 'STAT2', 'STAT1::STAT2', 'Stat2', 'Stat5a', 'Stat5b'],
    'AP1': ['JUN', 'Jun', 'JUNB', 'FOS', 'Fosb', 'BATF::JUN'],
    'TEA': ['TEAD1', 'TEAD2', 'TEAD4'],
}
all_targets = set()
for v in target_families.values():
    all_targets.update(v)

# Same-diff only, sorted by score (most negative = strongest)
sd = df[df.score < 0].copy()
sd = sd.sort_values('score')

# Count how many target families each seq has
def count_families(tfs_str):
    seq_tfs = set(tfs_str.split(','))
    return sum(1 for fam, members in target_families.items() if seq_tfs & set(members))

def which_families(tfs_str):
    seq_tfs = set(tfs_str.split(','))
    return [fam for fam, members in target_families.items() if seq_tfs & set(members)]

sd['n_families'] = sd.all_tfs.apply(count_families)
sd['families'] = sd.all_tfs.apply(which_families)
sd['has_target'] = sd.all_tfs.apply(lambda x: bool(set(x.split(',')) & all_targets))

# Prioritize: most families first, then most extreme score
sd_targets = sd[sd.has_target].sort_values(['n_families', 'score'], ascending=[False, True])

print(f'Same-diff sequences with target TFs: {len(sd_targets)}')
print(f'\nTop candidates (most families, most extreme score):')
for _, row in sd_targets.head(15).iterrows():
    print(f'  idx={row.idx:5d}  score={row.score:.3f}  r={row.corr:.3f}  ratio={row.ratio:.1f}  '
          f'families={row.families}  tfs={row.matching_tfs}')

Same-diff sequences with target TFs: 1297

Top candidates (most families, most extreme score):


TypeError: unsupported format string passed to method.__format__